In [ ]:
!pip install faiss-cpu sentence-transformers pandas numpy

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

In [ ]:
import pandas as pd

df = pd.read_excel("/content/ai_complex_dataset.xlsx")

df['Prompt'] = df['Prompt'].astype(str)
df['Cleaned'] = df['Prompt'].apply(clean_text)

print("Dataset loaded:", len(df))

In [ ]:
import pandas as pd

df = pd.read_excel("/content/ai_complex_dataset.xlsx")

df['Prompt'] = df['Prompt'].astype(str)
df['Cleaned'] = df['Prompt'].apply(clean_text)

print("Dataset loaded:", len(df))

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['Cleaned'].tolist())

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype('float32')

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS index ready!")

In [ ]:
restricted_keywords = [
    "hack password",
    "bypass security",
    "illegal access",
    "crack software",
    "generate malware",
    "ddos attack",
    "phishing attack"
]

def is_restricted(text):
    return any(word in text for word in restricted_keywords)

In [ ]:
restricted_keywords = [
    "hack password",
    "bypass security",
    "illegal access",
    "crack software",
    "generate malware",
    "ddos attack",
    "phishing attack"
]

def is_restricted(text):
    return any(word in text for word in restricted_keywords)

In [ ]:
valid_tasks_keywords = [
    "explain", "write", "generate", "create", "code",
    "image", "video", "audio", "calculate", "solve"
]

def is_valid_task(text):
    return any(word in text for word in valid_tasks_keywords)

In [ ]:
def safety_layer(user_input):

    text = user_input.lower()

    # Check restricted content
    if is_restricted(text):
        return {
            "status": "blocked",
            "message": "This request is restricted. Please try a valid AI task."
        }

    # Check valid task
    if not is_valid_task(text):
        return {
            "status": "invalid",
            "message": "Unsupported request. Try tasks like text, code, image, or calculation."
        }

    return {"status": "safe"}

In [ ]:
print(safety_layer("Generate malware code"))
print(safety_layer("Tell me a joke"))
print(safety_layer("Write Python code for sorting"))

In [ ]:
def ai_system(user_input, k=3):

    # STEP 5: Safety Layer
    safety = safety_layer(user_input)

    if safety["status"] != "safe":
        return safety

    # STEP 1: Clean input
    cleaned = clean_text(user_input)

    # STEP 2: Embedding
    query_embedding = model.encode([cleaned])
    query_embedding = np.array(query_embedding).astype('float32')

    # STEP 3: FAISS search
    distances, indices = index.search(query_embedding, k)

    # STEP 4: Get best match
    best_idx = indices[0][0]
    row = df.iloc[best_idx]

    return {
        "task": row["Task Type"],
        "primary_model": row["Primary Model"],
        "fallback_model": row["Fallback Model"]
    }

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def ui_system(user_input):

    result = ai_system(user_input)

    if result.get("status") == "blocked" or result.get("status") == "invalid":
        return result["message"]

    return f"""
    🧠 Task Detected: {result['task']}

    ⚡ Primary Model: {result['primary_model']}

    🚀 Fallback Model: {result['fallback_model']}
    """

In [ ]:
interface = gr.Interface(
    fn=ui_system,
    inputs=gr.Textbox(lines=3, placeholder="Enter your prompt here..."),
    outputs="text",
    title="🤖 AI Model Orchestration",
    description="Enter a prompt and get the best AI model suggestion"
)

interface.launch()

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# 🤖 AI Model Router")
    gr.Markdown("Enter a prompt and get best model suggestion")

    input_box = gr.Textbox(label="Your Prompt", lines=4)

    output_box = gr.Textbox(label="Result", lines=20)  # 🔥 increased height

    btn = gr.Button("Suggest Model")

    btn.click(fn=ui_system, inputs=input_box, outputs=output_box)

demo.launch()

In [ ]:
Edit → Clear all outputs